# Chapter 12: Monitoring and Scaling

*Deep Learning Crash Course - BPB Publications*

We build the data-drift detectors, monitoring metrics, scaling-strategy spreadsheet calculations, fairness audit utility, and retraining-trigger logic the chapter describes. Each section produces a small, self-contained artefact that can be wired into a real system.


## 1. Setup

In [ ]:
import os, json, time, random, hashlib
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

IMG_DIR = Path('images'); IMG_DIR.mkdir(exist_ok=True)
def set_seed(s=42):
    os.environ['PYTHONHASHSEED']=str(s); random.seed(s); np.random.seed(s)
set_seed(42)


## 2. KS test and PSI for feature-drift detection

The KS test compares cumulative distribution functions; PSI bins values and computes a weighted log-ratio.

In [ ]:
from scipy import stats

def psi(expected, actual, bins=10, eps=1e-6):
    edges = np.quantile(expected, np.linspace(0, 1, bins + 1))
    edges[0], edges[-1] = -np.inf, np.inf
    e_counts, _ = np.histogram(expected, edges); e_counts = e_counts / e_counts.sum() + eps
    a_counts, _ = np.histogram(actual, edges);   a_counts = a_counts / a_counts.sum() + eps
    return float(np.sum((a_counts - e_counts) * np.log(a_counts / e_counts)))

training = np.random.normal(0, 1, 5000)
no_drift = np.random.normal(0, 1, 5000)
small_drift = np.random.normal(0.2, 1, 5000)
big_drift = np.random.normal(1.0, 1, 5000)

for name, sample in [('no drift', no_drift), ('small drift', small_drift), ('big drift', big_drift)]:
    ks_stat, ks_p = stats.ks_2samp(training, sample)
    print(f'{name:13s} | KS stat {ks_stat:.3f} | KS p {ks_p:.3f} | PSI {psi(training, sample):.3f}')


In [ ]:
# Visualise distributions and their KS / PSI scores
fig, axes = plt.subplots(1, 3, figsize=(11, 3.5), sharey=True)
for ax, (name, sample) in zip(axes, [('no drift', no_drift), ('small drift', small_drift), ('big drift', big_drift)]):
    ax.hist(training, bins=40, alpha=0.4, label='training', density=True)
    ax.hist(sample, bins=40, alpha=0.4, label='live', density=True)
    ax.set_title(f'{name} | PSI {psi(training, sample):.2f}')
    ax.legend(fontsize=8)
fig.suptitle('Feature distributions before / after drift')
fig.tight_layout(); fig.savefig(IMG_DIR / '01_psi.png', dpi=150); plt.show()


## 3. CUSUM chart for detecting sudden changes

PSI and KS compare the whole-window distribution; CUSUM is sensitive to a sudden change in the mean.

In [ ]:
def cusum(values, target, k, h):
    pos = neg = 0
    s_pos, s_neg, alarm = [], [], []
    for v in values:
        pos = max(0, pos + (v - target - k))
        neg = min(0, neg + (v - target + k))
        s_pos.append(pos); s_neg.append(neg)
        alarm.append(pos > h or neg < -h)
    return np.array(s_pos), np.array(s_neg), np.array(alarm)

# Series: clean, then a shift
series = np.concatenate([np.random.normal(0.5, 0.1, 200),
                         np.random.normal(0.4, 0.1, 200)])
pos, neg, alarm = cusum(series, target=0.5, k=0.05, h=1.0)
fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
axes[0].plot(series); axes[0].axhline(0.5, color='gray', linestyle='--'); axes[0].set_ylabel('accuracy')
axes[1].plot(pos, label='pos'); axes[1].plot(neg, label='neg')
axes[1].axhline(1.0, color='red', linestyle='--'); axes[1].axhline(-1.0, color='red', linestyle='--')
axes[1].set_xlabel('step'); axes[1].set_ylabel('CUSUM'); axes[1].legend()
first_alarm = int(np.argmax(alarm)) if alarm.any() else -1
fig.suptitle(f'CUSUM alarm at step {first_alarm}')
fig.tight_layout(); fig.savefig(IMG_DIR / '02_cusum.png', dpi=150); plt.show()


## 4. Fairness audit - equalized odds

We compute true positive rate (TPR) and false positive rate (FPR) per group and flag when the gap exceeds a configurable threshold.

In [ ]:
def fairness_audit(y_true, y_pred, groups, threshold=0.05):
    metrics = {}
    for g in np.unique(groups):
        mask = groups == g
        yt, yp = y_true[mask], y_pred[mask]
        tp = int(((yp == 1) & (yt == 1)).sum())
        fn = int(((yp == 0) & (yt == 1)).sum())
        fp = int(((yp == 1) & (yt == 0)).sum())
        tn = int(((yp == 0) & (yt == 0)).sum())
        tpr = tp / max(tp + fn, 1)
        fpr = fp / max(fp + tn, 1)
        metrics[str(g)] = {'tpr': tpr, 'fpr': fpr, 'n': int(mask.sum())}
    tprs = [m['tpr'] for m in metrics.values()]
    fprs = [m['fpr'] for m in metrics.values()]
    gap_tpr = max(tprs) - min(tprs); gap_fpr = max(fprs) - min(fprs)
    return metrics, {'tpr_gap': gap_tpr, 'fpr_gap': gap_fpr,
                     'breaches_threshold': bool(max(gap_tpr, gap_fpr) > threshold)}

rng = np.random.default_rng(0)
n = 2000
groups = rng.choice(['A', 'B', 'C'], size=n)
y = rng.binomial(1, 0.3, size=n)
# A & B are well-calibrated; C is over-predicted (more false positives)
y_pred = y.copy()
for gi, prob_flip in [('A', 0.05), ('B', 0.05), ('C', 0.18)]:
    mask = (groups == gi) & (y == 0)
    flip = rng.binomial(1, prob_flip, mask.sum()).astype(bool)
    y_pred[np.where(mask)[0][flip]] = 1

metrics, summary = fairness_audit(y, y_pred, groups)
print(json.dumps(metrics, indent=2))
print('Summary:', summary)

# Bar chart
fig, ax = plt.subplots(figsize=(6.5, 3.5))
x = np.arange(len(metrics))
ax.bar(x - 0.2, [m['tpr'] for m in metrics.values()], width=0.4, label='TPR')
ax.bar(x + 0.2, [m['fpr'] for m in metrics.values()], width=0.4, label='FPR')
ax.set_xticks(x); ax.set_xticklabels(list(metrics.keys()))
ax.set_ylabel('rate'); ax.legend(); ax.set_title('Per-group TPR and FPR')
fig.tight_layout(); fig.savefig(IMG_DIR / '03_fairness.png', dpi=150); plt.show()


## 5. Audit record with content hash

The sha256 hash binds the metrics record to the exact model + dataset combo so auditors can verify no tampering happened later.

In [ ]:
def audit_record(model_id, dataset_id, metrics):
    payload = {
        'model_id': model_id,
        'dataset_id': dataset_id,
        'metrics': metrics,
        'timestamp': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    }
    h = hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()
    payload['sha256'] = h
    return payload

rec = audit_record('mnist-mlp@0.1.0', 'mnist-test-v2', metrics)
print(json.dumps(rec, indent=2))


## 6. Triggered-vs-scheduled retraining decision logic

In [ ]:
def retraining_decision(psi_value, accuracy_drop, confirmation_window_days, psi_threshold=0.2, accuracy_drop_threshold=0.02):
    """Return whether we should kick off a retraining run.

    A drift signal alone is not enough - we wait `confirmation_window_days` of sustained
    breach to avoid retraining on a transient anomaly (e.g. one bad batch).
    """
    confirmed = (psi_value > psi_threshold and confirmation_window_days >= 2) or (accuracy_drop > accuracy_drop_threshold)
    return {'retrain': bool(confirmed),
            'reason': ('accuracy degradation' if accuracy_drop > accuracy_drop_threshold else
                       'sustained drift' if confirmed else
                       'within tolerance')}

for psi_value, drop, window in [(0.05, 0.005, 1), (0.30, 0.005, 1), (0.30, 0.005, 3), (0.05, 0.04, 1)]:
    print(f'PSI={psi_value} drop={drop} window={window} -> {retraining_decision(psi_value, drop, window)}')


## 7. Scaling cost estimation

Simple spreadsheet calculation: given peak/trough RPS, target latency, instance throughput and instance price, compute monthly cost for three strategies.

In [ ]:
def monthly_cost(min_inst, max_inst, fraction_at_max, hourly_price):
    hours_month = 24 * 30
    avg = fraction_at_max * max_inst + (1 - fraction_at_max) * min_inst
    return avg * hourly_price * hours_month

peak_rps, trough_rps = 800, 50
per_inst_capacity = 100  # requests / second / instance at SLA
needed_max = int(np.ceil(peak_rps / per_inst_capacity))
needed_min = max(2, int(np.ceil(trough_rps / per_inst_capacity)))

strategies = [
    ('Fixed peak GPU fleet',  needed_max, needed_max, 1.0,  3.50),
    ('KEDA-scaled spot CPU',  needed_min, needed_max, 0.10, 0.30),
    ('Serverless first',      0,          needed_max, 0.05, 0.45),
]
for label, mn, mx, frac_max, price in strategies:
    cost = monthly_cost(mn, mx, frac_max, price)
    print(f'{label:25s} min={mn:2d} max={mx:2d} cost=${cost:8.2f}/month')


## 8. Putting it together - a monitoring dashboard sketch

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 6))
axes[0, 0].plot(np.cumsum(np.random.randn(100)*0.02) + 0.91); axes[0, 0].set_title('Accuracy (rolling 1h)')
axes[0, 0].set_ylim(0.7, 1.0); axes[0, 0].axhline(0.88, color='red', ls='--')
axes[0, 1].plot(np.linspace(15, 75, 100) + np.random.randn(100)*3); axes[0, 1].set_title('p99 latency (ms)')
axes[0, 1].axhline(80, color='red', ls='--')
axes[1, 0].plot([psi(training, np.random.normal(i / 50, 1, 1000)) for i in range(100)])
axes[1, 0].set_title('Feature PSI (top feature)'); axes[1, 0].axhline(0.2, color='red', ls='--')
axes[1, 1].plot(50 + np.cumsum(np.random.randn(100) * 5)); axes[1, 1].set_title('RPS')
for ax in axes.flat: ax.set_xlabel('minute'); ax.grid(alpha=0.3)
fig.suptitle('Live monitoring dashboard - sketch'); fig.tight_layout()
fig.savefig(IMG_DIR / '04_dashboard.png', dpi=150); plt.show()


## 9. Exercise solutions

### 9.1 MCQ answer key

| Q | Answer | Why |
|---|--------|-----|
| 1 | (c) Material drift (>0.2) | PSI threshold from the chapter. |
| 2 | (b) Capture late-arriving ground-truth labels for monitoring | Closes the feedback loop. |
| 3 | (b) Stitch together causality across services | Logs are per-event; traces follow a request. |
| 4 | (b) Custom metrics (queue depth, RPS, GPU util) | KEDA's USP. |
| 5 | (b) Premature scale-downs during transient dips | Stabilisation window. |
| 6 | (b) Returns cached predictions for repeat inputs | Memoisation. |
| 7 | (b) Same TPR and FPR | Equalized-odds definition. |
| 8 | (b) Integrity / tamper-evidence | Hash binds metrics to model and data. |
| 9 | (b) Detects state-distribution drift | V(s) shifts when the state distribution changes. |
| 10 | (b) Only retrains when needed | Saves compute and prevents unnecessary churn. |
| 11 | (b) Retraining on transient anomalies | Confirmation window prevents false positives. |
| 12 | (c) State value V(s) | Always-on, can be computed for every observation. |


### 9.2 12% confidence drop on day 45
A confidence drop without a latency or error-rate change is most consistent with **data drift** or a **monitoring bug**. Diagnostic sequence:
1. **Reproduce on a sample of recent requests**: pull 1k recent inputs, run them through both the current and previous model versions. If confidence drop reproduces -> drift; if confidence is identical -> instrumentation bug.
2. **PSI on each input feature** (section 2). PSI > 0.2 on any feature confirms drift.
3. **Compare per-class prediction distributions** to the training set. A different class mix can drop average confidence even with no per-class accuracy change.
4. **Inspect the monitoring pipeline**: was the model build digest stable? Did the runtime change? A library upgrade can quietly change softmax temperature.
5. **If labels are available**: compute current per-class accuracy. Confidence drop with stable accuracy = label-shift; confidence drop with accuracy drop = full degradation -> retrain.

### 9.3 Fraud-detection monitoring strategy
**Latency SLA: sub-100ms p99 at 5,000 RPS**
- Metric: `fraud_inference_latency_seconds` (Prometheus histogram), alert if 5-min p99 > 95 ms.
- Metric: `fraud_inference_throughput`, alert if RPS drops > 30 percent below the rolling baseline.

**Two-day label lag**:
- Metric: `fraud_predictions_with_labels{age='48h'}`, computed on the labels-arrived stream.
- Metric: `fraud_recall_48h` and `fraud_precision_48h`, alert on > 2 sigma deviation from the 14-day rolling mean.
- Use a shadow champion-challenger comparison since point-estimates are noisy.

**Equal-Credit-Opportunity Act compliance**:
- Per-protected-group TPR, FPR, AUC; weekly fairness audit (section 4); persist a signed audit record (section 5).

**Escalation**:
- Latency: page on-call SRE.
- Accuracy drop: page model owner.
- Fairness breach: notify legal + compliance and freeze deploys.

### 9.4 KS vs PSI vs CUSUM
- **KS test** is a non-parametric, two-sample comparison of cumulative distributions. Detects any distributional change; needs both samples in memory; gives a p-value.
- **PSI** is a binned, weighted log-ratio. Operates on histograms so it scales to billion-row streams via approximate quantile sketches.
- **CUSUM** is a sequential mean-shift detector. Detects sudden shifts faster than either KS or PSI because it integrates evidence over time.

**Multi-layer pipeline (100k requests/day)**:
- CUSUM on rolling 1-hour mean of each feature -> detects sudden shifts within hours.
- PSI nightly on the previous 24h vs the training window -> catches gradual drift.
- KS weekly on the previous 7d vs training -> highest power, catches subtle distribution-shape changes.
- Each layer writes an alert; only flag for retraining when at least two layers fire within the same week.

### 9.5 KEDA scale-out trace
Configuration: `RPS/pod > 50`, `minReplicas=2`, `maxReplicas=20`. Current: 6 pods, 380 RPS (63 RPS/pod). Spike to 1,200 RPS (200 RPS/pod) over 30 s.
1. **t=0** KEDA polling sees 200 RPS/pod, target 50 -> scale ratio 4x. Computes desired 24 pods, capped at 20.
2. **t=0..15 s** Kubernetes adds 14 pods in batches of 4 (default `behavior.scaleUp.policies`).
3. **t=15..45 s** New pods start; image pull + lifespan warmup ~10-20 s each.
4. **t=45..120 s** Once pods are Ready, traffic spreads; per-pod load drops to 60 RPS/pod.
5. **Latency** during the transition: existing pods are saturated, p99 spikes to 200-300 ms while new pods are starting. To reduce: (a) **lower the scale-up stabilization window**, (b) **pre-pull the image** with a DaemonSet, (c) **enable HPA's `predictiveAutoscaling`** to scale on rate-of-change rather than absolute level, (d) **provision warm pool** with `minReplicas=6` so the cold start hits a smaller delta.

### 9.6 PPO PSI 0.35 without return degradation
Three possible explanations:
1. **State distribution drift** caused by environmental changes (e.g. new product mix in the warehouse). Policy still completes tasks because they remain in the policy's competence envelope.
2. **Reward distribution drift** while the task definition is unchanged - the agent receives the same nominal reward but the rate of high-value episodes shifts. The action histogram looks different but task success is unchanged.
3. **Sensor calibration drift** changes the observed state without changing the underlying physical system. The agent's policy is observationally different but produces the same physical action.

**Diagnostic**:
- Compare action-by-state distribution between current and training data. If actions for the same state are unchanged -> only state drift. If actions for the same state have shifted -> policy is responding to drift -> closer to a real issue.
- Inspect raw sensor calibration logs.

**Rollback criteria** despite good task performance: rollback if (a) policy starts producing actions outside training-set range (extrapolation risk), (b) safety-critical envelopes are violated even if task succeeds, (c) the value function is significantly more uncertain than during training (high TD error) - the policy may be one bad state away from failure.

### 9.7 Stratified sampling for TPR gap
TPR (recall) is a per-positive metric. A gap means the model finds positives in one group less often than in another. **Stratified sampling** during retraining means drawing positive examples for each demographic group with equal weight rather than naturally - so the under-represented group's positive class is no longer drowned out by the majority.

**Why it addresses TPR specifically**: TPR responds primarily to how well the positive class is fit. Stratification at the *positive* level raises the model's exposure to under-represented positives -> closes the recall gap.

**Risk of over-correction**: severe up-weighting can over-fit the minority positive class and harm minority precision (more false positives) and overall calibration.

**Two alternative interventions**:
1. **Threshold adjustment per group** - keep the model unchanged but lower the decision threshold for the under-served group until TPR equalises. Cheaper, but creates per-group thresholds that need audit transparency.
2. **Auxiliary fairness loss** - add an MMD-style term between per-group output distributions during training, encouraging similar predictions without changing the data.

### 9.8 Three cost strategies for 800 RPS peak / 50 RPS trough
Using the calculator in section 7:
- (a) **Fixed on-demand GPU fleet at peak**: 8 instances at \$3.50/hr * 720h = **\$20,160/month**. Risk: zero capacity headroom in a spike.
- (b) **KEDA-scaled spot CPU + 30 percent on-demand fallback**: avg 2 to 8 instances, blended price ~\$0.50/hr -> **\$2,200/month**. Risk: spot reclaim can cause sudden capacity loss; need a healthy on-demand baseline.
- (c) **Serverless-first with on-demand fallback above 100 RPS**: dominant cost is the burst tier above 100 RPS, which for a 90/10 split is ~\$5,000/month. Risk: cold-start latency spikes on traffic ramp.
Recommendation: strategy (b) is the cheapest, but needs a clear capacity-floor and clear monitoring of spot reclaim events.

---
*End of Chapter 12 - end of book bundle.*
